# UdaciMed | Notebook 3: Hardware Acceleration & Production Deployment

Welcome to the final phase of UdaciMed's optimization pipeline! In this notebook, you will implement cross-platform hardware acceleration techniques and strategize for the deployment of your optimized model across hardware targets.

## Recap: Optimization Journey

In [Notebook 2](02_architecture_optimization.ipynb), you have implemented architectural optimizations that brought you closer to your optimization targets.

Now, it is time to unlock further performance opportunities with hardware acceleration.

> **Your mission**: Transform your optimized model into a production-ready cross-platform deployment that meets production SLAs on this reference hardware, and finalize UdaciMed's deployment strategy across its diverse hardware fleet.

### Hardware acceleration

You will implement and evaluate **2 core deployment techniques\*** using [ONNX Runtime](https://onnxruntime.ai/):

1. **Mixed Precision (FP16)** - Utilizing 16-bit floating-point numbers to significantly speed up calculations and reduce memory usage on compatible hardware.
2. **Dynamic Batching** - Finding the best batch size to maximize throughput for offline tasks while maintaining low latency for real-time requests.

Additionally, you will analyze three deployment scenarios: GPU (TensorRT), CPU (OpenVINO), and Edge deployment considerations.

_\* Note that while you are expected to implement both deployment techniques, you can decide whether to keep either or both in your final deployment strategy to best achieve targets._

---

Through this notebook, you will:

- **Convert PyTorch model to ONNX** for cross-platform deployment
- **Apply hardware acceleration using ONNX Runtime** on the reference T4 device
- **Benchmark end-to-end performance** against SLAs
- **Validate clinical safety** across the deployment pipeline
- **Analyze alternative deployment strategies** for diverse hardware environments

**Let's deliver a production-ready, hardware-accelerated diagnostic deployment!**

## Step 1: Setup the environment

First, let's set up the environment and understand our reference hardware capabilities. 

This ensures our optimization and benchmarking code will run smoothly.

In [1]:
# Use the CUDA 12 build; the default PyPI 1.18.0 wheel expects CUDA 11 libraries.
! pip install --user --force-reinstall --no-deps --index-url "https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/" "onnxruntime-gpu==1.18.0"
! pip install --user nvidia-ml-py

Looking in indexes: https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.6/200.6 MB 5.4 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: onnxruntime-gpu
    Found existing installation: onnxruntime-gpu 1.18.0
    Uninstalling onnxruntime-gpu-1.18.0:
      Successfully uninstalled onnxruntime-gpu-1.18.0


In [2]:
! nvidia-smi --query-gpu=index,uuid,memory.used --format=csv
! nvidia-smi --query-compute-apps=gpu_uuid,pid,process_name,used_gpu_memory --format=csv

index, uuid, memory.used [MiB]
0, GPU-949bb6b5-49d4-72dd-1028-3d08f4321b0f, 0 MiB
gpu_uuid, pid, process_name, used_gpu_memory [MiB]


In [3]:
# Import PyTorch first to preload its CUDA 12/cuDNN 8 libraries before ORT creates a session.
import torch
import onnxruntime as ort

print(f"PyTorch CUDA build: {torch.version.cuda}")
print(f"ONNX Runtime version: {ort.__version__}")
print(f"ONNX Runtime package: {ort.__file__}")
available_providers = ort.get_available_providers()
print(f"Available ONNX Runtime providers: {available_providers}")
if "CUDAExecutionProvider" not in available_providers:
    raise RuntimeError(
        "CUDAExecutionProvider is missing. Verify the CUDA 12 ONNX Runtime build was installed and restart the kernel."
    )

PyTorch CUDA build: 12.1
ONNX Runtime version: 1.18.0
ONNX Runtime package: /voc/work/.local/lib/python3.10/site-packages/onnxruntime/__init__.py
Available ONNX Runtime providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [4]:
# Make sure that libraries are dynamically re-loaded if changed
%load_ext autoreload
%autoreload 2

In [5]:
# Import core libraries
import torch
import torch.nn as nn
import numpy as np
import onnx
import onnxruntime as ort
import pickle
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any, Literal
import warnings
warnings.filterwarnings('ignore')

# Import project utilities
from utils.data_loader import (
    load_pneumoniamnist,
    get_sample_batch
)
from utils.model import (
    create_baseline_model,
    get_model_info
)
from utils.evaluation import (
    evaluate_with_multiple_thresholds
)
from utils.profiling import (
    PerformanceProfiler,
    measure_time
)
from utils.visualization import (
    plot_performance_profile,
    plot_batch_size_comparison
)
from utils.architecture_optimization import (
    create_optimized_model
)

In [6]:
# Set device and analyze hardware capabilities
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    
    # Check tensor core support for mixed precision - crucial for FP16 acceleration
    gpu_compute = torch.cuda.get_device_properties(0).major
    tensor_core_support = gpu_compute >= 7  # Volta+ architecture
    print(f"Tensor Core Support: {tensor_core_support}")
else:
    print("WARNING: CUDA not available - hardware acceleration will be limited")

print("Default hardware acceleration environment ready!")

# Verify ONNX Runtime GPU support
print(f"\nONNX Runtime available providers: {ort.get_available_providers()}")

Using device: cuda
GPU: Tesla T4
GPU Memory: 14.6 GB
Tensor Core Support: True
Default hardware acceleration environment ready!

ONNX Runtime available providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


> **Getting ready for acceleration**: The checks above highlight two critical facts for our mission:
> 1. Our reference hardware has tensor core support, which can dramatically speed up 16-bit floating-point (FP16) calculations; for other hardware deployments, like CPUs that lack this feature, we would need to rely on different techniques (such as 8-bit integer quantization (INT8)) to achieve similar acceleration.
> 2. ONNX Runtime providers are available for our primary targets: CUDAExecutionProvider for GPU and CPUExecutionProvider for CPU. This allows us to benchmark on both platforms. For a true mobile or edge deployment, we would need to use a specialized package like ONNX Runtime Mobile, which is built separately to keep the application lightweight.
> 
> Our task is to meet SLAs on our current device, which means we must **_benchmark against the GPU_** to see if we've met our goals.

## Step 2: Load test data and optimized model with configuration

The model is needed for deployment, and the optimization results for comparison.

Test data is needed for both conversion and final performance testing.

In [7]:
# Define dataset loading parameters
img_size = 64
batch_size = 32

# Load test dataset for final evaluation
test_loader = load_pneumoniamnist(
    split="test", 
    download=True, 
    size=img_size,
    batch_size=batch_size,
    subset_size=None
)

# Get sample batch for profiling
sample_images, sample_labels = get_sample_batch(test_loader)
sample_images = sample_images.to(device)
sample_labels = sample_labels.to(device)

print(f"Test data loaded: {sample_images.shape} batch for hardware acceleration profiling")

Using downloaded and verified file: /voc/work/.medmnist/pneumoniamnist_64.npz
Test data loaded: torch.Size([32, 3, 64, 64]) batch for hardware acceleration profiling


> **Batch size strategy**: Your batch size choice impacts memory usage, latency, and throughput. 
> 
> Consider: What batch size best applied for each deployment scenario? Don't forget to review the batch analysis plot from Notebook 2!

In [8]:
# Load optimized model and results from notebook 2

# Resolve experiment name from available result artifacts (latest file wins).
results_dir = Path("../results")
result_files = sorted(results_dir.glob("optimization_results_*.pkl"), key=lambda p: p.stat().st_mtime)
if not result_files:
    raise FileNotFoundError(
        "No optimization results found. Run Notebook 2 first to generate optimization_results_*.pkl"
    )

latest_results_path = result_files[-1]
experiment_name = latest_results_path.stem.replace("optimization_results_", "")

with open(latest_results_path, "rb") as f:
    optimization_results = pickle.load(f)

print("Loaded optimization results from Notebook 2:")
print(f"   Experiment: {experiment_name}")
print(f"   Model: {optimization_results['model_name']}")
print(f"   Clinical Performance: {optimization_results['clinical_performance']['optimized']['sensitivity']:.1%} sensitivity")
print(f"   Architecture Speedup: {optimization_results['performance_improvements']['latency_speedup']:.2f}x")
print(f"   Memory Reduction: {optimization_results['performance_improvements']['memory_reduction_percent']:.1f}%")

Loaded optimization results from Notebook 2:
   Experiment: optimized_interpolation_removal_depthwise_separable_channel_optimization_use_amp
   Model: ResNet-18 Optimized
   Clinical Performance: 99.0% sensitivity
   Architecture Speedup: 0.62x
   Memory Reduction: 75.0%


> **HINT: Finding your optimization results**
> 
> Your optimization results from Notebook 2 should be saved as:
> - Results file: `../results/optimization_results_{experiment_name}.pkl`
> - Model weights: `../results/optimized_model.pth`
> 
> The experiment name typically combines your optimization techniques, like:
> - `"interpolation-removal_depthwise-separable"`
> - `"channel-reduction_grouped-conv"`

In [9]:
# Get the optimization configuration
opt_config = optimization_results['optimization_config']
optimized_model = None

# Rebuild the architecture exactly as in notebook 2, then load trained optimized weights.
baseline_for_load = create_baseline_model(
    num_classes=2,
    input_size=img_size,
    pretrained=False,
    fine_tune=True,
)
optimized_model = create_optimized_model(baseline_for_load, opt_config)

weights_path = Path("../results/optimized_model.pth")
if not weights_path.exists():
    raise FileNotFoundError(
        f"Optimized weights not found at {weights_path}. Run Notebook 2 save step first."
    )

state_dict = torch.load(weights_path, map_location=device)
optimized_model.load_state_dict(state_dict, strict=False)
optimized_model = optimized_model.to(device)
optimized_model.eval()

print("Optimized model loaded and ready for ONNX export")
print(f"   Weights path: {weights_path}")
print(f"   Device: {device}")

Starting clinical model optimization pipeline...
   Applying interpolation removal optimization...
Applying native resolution optimization (64x64)...
INTERPOLATION REMOVAL completed.
   Applying depthwise separable optimization...
Applying depthwise separable convolution optimization...
DEPTHWISE SEPARABLE completed: Successfully applied to layers with 16 replacements
   Applying channel optimization optimization...
Applying channel-level hardware optimizations...
   Converted 0 activation(s) to in-place
   Converted model to channels_last memory format
CHANNEL OPTIMIZATION completed
Applied optimizations in order: interpolation_removal → depthwise_separable → channel_optimization
Optimized model loaded and ready for ONNX export
   Weights path: ../results/optimized_model.pth
   Device: cuda


## Step 3: Convert model with hardware acceleration for production deployment

Convert the optimized model to [ONNX (Open Neural Network Exchange)](https://onnx.ai/) with optional hardware accelerations. 

**IMPORTANT**: You are tasked to implement both hardware optimizations even if you decide to disable them for the final export.

In [10]:
# Define deployment configuration for ONNX export.
# FP16 is enabled only when CUDA is available; dynamic batching helps support variable request sizes.
use_fp16 = torch.cuda.is_available()
use_dynamic_batching = True

print("Deployment export configuration:")
print(f"   FP16 enabled: {use_fp16}")
print(f"   Dynamic batching: {use_dynamic_batching}")

Deployment export configuration:
   FP16 enabled: True
   Dynamic batching: True


In [11]:
# Convert PyTorch model to ONNX format (for cross-platform deployment)

def export_model_to_onnx(model: nn.Module, input_tensor: torch.Tensor,
                        export_path: str, model_name: str = "pneumonia_detection",
                        fp16_mode: bool = use_fp16, dynamic_batching: bool = use_dynamic_batching) -> str:
    """
    Export PyTorch model to ONNX format for production deployment.
    Apply hardware optimizations if selected.

    Args:
        model: PyTorch model to export
        input_tensor: Sample input tensor for shape inference
        export_path: Directory to save the ONNX model
        model_name: Name for the exported ONNX file
        fp16_mode: If True, exports the model in FP16 (mixed precision)
        dynamic_batching: If True, configures the model to accept variable batch sizes

    Returns:
        Path to exported ONNX model
    """
    # Define output path, and ensure it exists
    onnx_path = f"{export_path}/{model_name}.onnx"
    Path(export_path).mkdir(parents=True, exist_ok=True)

    # Keep export dtype consistent between weights and example input.
    model.eval()
    export_input = input_tensor
    export_model = model.to(input_tensor.device)

    if fp16_mode:
        if not torch.cuda.is_available():
            print("WARNING: FP16 requested but CUDA unavailable; falling back to FP32 export")
            fp16_mode = False
        else:
            export_model = export_model.half()
            export_input = export_input.half()
    else:
        export_model = export_model.float()
        export_input = export_input.float()

    print(f"Exporting model to ONNX format...")
    print(f"   Input shape: {export_input.shape}")
    print(f"   Input dtype: {export_input.dtype}")
    print(f"   FP16 mode: {fp16_mode}")
    print(f"   Dynamic batching: {dynamic_batching}")
    print(f"   Export path: {onnx_path}")

    dynamic_axes = None
    if dynamic_batching:
        dynamic_axes = {
            "input": {0: "batch_size"},
            "output": {0: "batch_size"},
        }

    # Export to ONNX format with defined parameters
    torch.onnx.export(
        export_model,
        export_input,
        onnx_path,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes=dynamic_axes,
        opset_version=16,
        do_constant_folding=True,
        verbose=False,
    )

    print(f"ONNX export completed: {onnx_path}")

    # Verify ONNX model integrity - sanity check
    try:
        onnx_model = onnx.load(onnx_path)
        onnx.checker.check_model(onnx_model)
        print("   ONNX model verification passed")
    except Exception as e:
        print(f"   WARNING: ONNX verification failed: {str(e)}")

    # Restore model precision for potential downstream PyTorch use.
    if fp16_mode:
        model.float()

    return onnx_path

# Export the model to ONNX
onnx_model_path = export_model_to_onnx(
    model=optimized_model,
    input_tensor=sample_images,
    export_path="../results/onnx_models",
    model_name="udacimed_pneumonia_optimized",
)
print(f"Final ONNX artifact: {onnx_model_path}")

Exporting model to ONNX format...
   Input shape: torch.Size([32, 3, 64, 64])
   Input dtype: torch.float16
   FP16 mode: True
   Dynamic batching: True
   Export path: ../results/onnx_models/udacimed_pneumonia_optimized.onnx
ONNX export completed: ../results/onnx_models/udacimed_pneumonia_optimized.onnx
   ONNX model verification passed
Final ONNX artifact: ../results/onnx_models/udacimed_pneumonia_optimized.onnx


In [12]:
# Release temporary PyTorch GPU tensors before measuring ONNX Runtime process memory.
optimized_model = None
baseline_for_load = None
state_dict = None
sample_images = sample_images.detach().cpu()
sample_labels = sample_labels.detach().cpu()
if torch.cuda.is_available():
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
print("Released temporary PyTorch model allocations; benchmark inputs retained on CPU.")

Released temporary PyTorch model allocations; benchmark inputs retained on CPU.


## Step 4: Deploy with ONNX Runtime

With our model saved in the ONNX format, we can now load it into the [ONNX Runtime (ORT)](https://onnxruntime.ai/getting-started). 

ORT is a high-performance inference engine that can execute models on different hardware backends through its **Execution Providers (EPs)**. 

In [13]:
# This function creates an ONNX Runtime Inference Session.

use_gpu = torch.cuda.is_available()
cuda_memory_options = {
    "arena_extend_strategy": "kSameAsRequested",
    "cudnn_conv_algo_search": "HEURISTIC",
    "cudnn_conv_use_max_workspace": "0",
    "gpu_mem_limit": str(64 * 1024 * 1024),
}

def create_inference_session(model_path: str, use_gpu: bool = use_gpu) -> ort.InferenceSession:
    """Create an ORT session and verify requested CUDA memory settings took effect."""
    print(f"Creating ONNX Runtime session for {'GPU' if use_gpu else 'CPU'}...")
    available_providers = ort.get_available_providers()
    providers = ["CPUExecutionProvider"]
    if use_gpu and "CUDAExecutionProvider" in available_providers:
        providers = [("CUDAExecutionProvider", dict(cuda_memory_options)), "CPUExecutionProvider"]

    session_options = ort.SessionOptions()
    session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    session = ort.InferenceSession(model_path, sess_options=session_options, providers=providers)

    active_providers = session.get_providers()
    print(f"Session created with providers: {active_providers}")
    if use_gpu and "CUDAExecutionProvider" not in active_providers:
        raise RuntimeError(
            "CUDAExecutionProvider failed to initialize; refusing to report CPU fallback performance as GPU results."
        )
    if use_gpu:
        active_options = session.get_provider_options().get("CUDAExecutionProvider", {})
        print(f"Active CUDA provider options: {active_options}")
        mismatches = {
            key: {"requested": value, "active": active_options.get(key)}
            for key, value in cuda_memory_options.items()
            if str(active_options.get(key)) != str(value)
        }
        if mismatches:
            raise RuntimeError(
                f"CUDA memory options were not applied: {mismatches}. "
                "Do not benchmark this session; recreate it with the requested provider options."
            )
        print("Verified CUDA arena limit: 64 MiB; total NVML process memory also includes CUDA context and libraries.")
    return session

# Create the session for our exported ONNX model.
inference_session = create_inference_session(onnx_model_path)

Creating ONNX Runtime session for GPU...
Session created with providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']
Active CUDA provider options: {'prefer_nhwc': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_enable': '0', 'enable_cuda_graph': '0', 'tunable_op_max_tuning_duration_ms': '0', 'tunable_op_tuning_enable': '0', 'cudnn_conv_use_max_workspace': '0', 'use_tf32': '1', 'cudnn_conv1d_pad_to_nc1d': '0', 'do_copy_in_default_stream': '1', 'cudnn_conv_algo_search': 'HEURISTIC', 'gpu_external_empty_cache': '0', 'gpu_external_free': '0', 'gpu_external_alloc': '0', 'gpu_mem_limit': '67108864', 'arena_extend_strategy': 'kSameAsRequested', 'user_compute_stream': '0', 'has_user_compute_stream': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0'}
Verified CUDA arena limit: 64 MiB; total NVML process memory also includes CUDA context and libraries.


In [14]:
print(inference_session.get_provider_options())

{'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'prefer_nhwc': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_enable': '0', 'enable_cuda_graph': '0', 'tunable_op_max_tuning_duration_ms': '0', 'tunable_op_tuning_enable': '0', 'cudnn_conv_use_max_workspace': '0', 'use_tf32': '1', 'cudnn_conv1d_pad_to_nc1d': '0', 'do_copy_in_default_stream': '1', 'cudnn_conv_algo_search': 'HEURISTIC', 'gpu_external_empty_cache': '0', 'gpu_external_free': '0', 'gpu_external_alloc': '0', 'gpu_mem_limit': '67108864', 'arena_extend_strategy': 'kSameAsRequested', 'user_compute_stream': '0', 'has_user_compute_stream': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0'}}


# Step 5: Benchmark model performance on all metrics

Now that we have a hardware-accelerated inference session, it's time to measure its performance. 

Unlike a server-based approach, we will perform direct, client-side benchmarking. This gives us precise measurements of the model's raw inference speed and resource consumption on our target hardware.

In [15]:
# Define a helper function to get input details and type

def get_input_details(session: ort.InferenceSession) -> Tuple[str, Tuple, np.dtype]:
    """
    Gets the input name, shape, and dtype for an ONNX Runtime session.
    """
    input_details = session.get_inputs()[0]
    input_name = input_details.name

    # Match NumPy dtype to ONNX input tensor dtype.
    input_type = str(input_details.type).lower()
    is_fp16 = "float16" in input_type

    # Determine the correct numpy dtype
    input_dtype = np.float16 if is_fp16 else np.float32

    return input_name, input_details.shape, input_dtype

In [16]:
# This is the main benchmarking function.

import os
import threading


def resolve_nvml_process_id(session, pynvml, device_handles) -> Optional[int]:
    """Resolve a hidden host PID using two allocation/release correlation checks."""
    def process_memory_snapshot():
        memory_by_pid = {}
        for device_handle in device_handles:
            for process in pynvml.nvmlDeviceGetComputeRunningProcesses(device_handle):
                used_bytes = process.usedGpuMemory
                if used_bytes is not None and 0 <= int(used_bytes) < (1 << 63):
                    memory_by_pid[process.pid] = memory_by_pid.get(process.pid, 0) + int(used_bytes)
        return memory_by_pid

    try:
        snapshot = process_memory_snapshot()
        if os.getpid() in snapshot:
            return os.getpid()
        if not torch.cuda.is_available():
            return None
        provider_options = session.get_provider_options().get("CUDAExecutionProvider", {})
        cuda_device = int(provider_options.get("device_id", 0))
        probe_bytes = 32 * 1024 * 1024
        candidates = None
        with torch.cuda.device(cuda_device):
            for _ in range(2):
                torch.cuda.synchronize()
                torch.cuda.empty_cache()
                before = process_memory_snapshot()
                probe_tensor = None
                try:
                    probe_tensor = torch.empty(probe_bytes, dtype=torch.uint8, device=f"cuda:{cuda_device}")
                    probe_tensor.fill_(0)
                    torch.cuda.synchronize()
                    during = process_memory_snapshot()
                finally:
                    probe_tensor = None
                    torch.cuda.empty_cache()
                    torch.cuda.synchronize()
                after = process_memory_snapshot()
                matching_pids = {
                    process_id for process_id, during_bytes in during.items()
                    if process_id in before and process_id in after
                    and abs((during_bytes - before[process_id]) - probe_bytes) <= 4 * 1024 * 1024
                    and abs((during_bytes - after[process_id]) - probe_bytes) <= 4 * 1024 * 1024
                }
                candidates = matching_pids if candidates is None else candidates & matching_pids
        if candidates and len(candidates) == 1:
            host_pid = candidates.pop()
            print(f"NVML host PID {host_pid} correlated with notebook PID {os.getpid()} using two allocation/release probes.")
            return host_pid
        print("WARNING: GPU allocation probes did not identify a unique host PID; no process will be guessed.")
    except (pynvml.NVMLError, RuntimeError) as exc:
        print(f"WARNING: Host PID resolution failed: {exc}")
    return None


def measure_peak_gpu_process_memory(session: ort.InferenceSession,
                                   input_name: str,
                                   output_name: str,
                                   input_array: np.ndarray,
                                   memory_probe_runs: int = 25,
                                   sample_interval_seconds: float = 0.005) -> Optional[float]:
    """Sample resolved process GPU memory; PID-resolution probes are excluded."""
    if "CUDAExecutionProvider" not in session.get_providers():
        return None

    try:
        import pynvml
    except ImportError:
        print("WARNING: Install nvidia-ml-py; GPU process memory is unverified.")
        return None

    initialized = False
    try:
        pynvml.nvmlInit()
        initialized = True
        device_handles = [
            pynvml.nvmlDeviceGetHandleByIndex(index)
            for index in range(pynvml.nvmlDeviceGetCount())
        ]
    except pynvml.NVMLError as exc:
        if initialized:
            pynvml.nvmlShutdown()
        print(f"WARNING: NVML unavailable ({exc}); GPU process memory is unverified.")
        return None

    try:
        process_id = resolve_nvml_process_id(session, pynvml, device_handles)
        if process_id is None:
            return None
        stop_monitor = threading.Event()
        monitor_errors = []
        peak_memory_bytes = 0
        valid_samples = 0

        def sample_process_memory() -> None:
            nonlocal peak_memory_bytes, valid_samples
            total_memory_bytes = 0
            matched_process = False
            try:
                for device_handle in device_handles:
                    processes = pynvml.nvmlDeviceGetComputeRunningProcesses(device_handle)
                    matched_memory = [
                        int(process.usedGpuMemory)
                        for process in processes
                        if process.pid == process_id
                        and process.usedGpuMemory is not None
                        and 0 <= int(process.usedGpuMemory) < (1 << 63)
                    ]
                    if matched_memory:
                        total_memory_bytes += max(matched_memory)
                        matched_process = True
                if matched_process:
                    valid_samples += 1
                    peak_memory_bytes = max(peak_memory_bytes, total_memory_bytes)
            except pynvml.NVMLError as exc:
                monitor_errors.append(str(exc))

        sample_process_memory()

        def monitor_memory() -> None:
            while not stop_monitor.wait(sample_interval_seconds):
                sample_process_memory()

        monitor_thread = threading.Thread(target=monitor_memory, daemon=True)
        monitor_thread.start()
        try:
            for _ in range(memory_probe_runs):
                session.run([output_name], {input_name: input_array})
                sample_process_memory()
        finally:
            stop_monitor.set()
            monitor_thread.join()

        sample_process_memory()
        if monitor_errors or valid_samples == 0:
            detail = monitor_errors[0] if monitor_errors else (
                f"No valid NVML memory samples for resolved PID {process_id}."
            )
            print(f"WARNING: {detail} GPU process memory is unverified; other benchmarks continue.")
            return None
        return peak_memory_bytes / (1024 * 1024)
    finally:
        pynvml.nvmlShutdown()


def benchmark_performance(session: ort.InferenceSession,
                          test_data: torch.Tensor,
                          batch_sizes: List[int],
                          num_runs: int = 1000) -> Dict[str, Any]:
    """
    Benchmarks mean, p95, and p99 latency, throughput, and process GPU memory.

    Args:
        session: The ONNX Runtime inference session.
        test_data: A batch of test data for inference.
        batch_sizes: A list of batch sizes to test.
        num_runs: Number of timed inference runs.

    Returns:
        Per-batch mean/p95/p99 latency, throughput, and process GPU memory in MB.
    """
    results = {}
    output_name = session.get_outputs()[0].name
    input_name, _, input_dtype = get_input_details(session)
    print(f"Benchmarking with input dtype: {input_dtype}")
    print(f"Timed latency samples per batch size: {num_runs}")

    for batch_size in batch_sizes:
        print(f"--- Benchmarking Batch Size: {batch_size} ---")
        input_array = test_data[:batch_size].detach().cpu().numpy().astype(input_dtype)

        for _ in range(10):
            session.run([output_name], {input_name: input_array})

        latencies = []
        for _ in range(num_runs):
            start_time = time.perf_counter()
            session.run([output_name], {input_name: input_array})
            latencies.append((time.perf_counter() - start_time) * 1000)

        p95_latency_ms = float(np.percentile(latencies, 95))
        p99_latency_ms = float(np.percentile(latencies, 99))

        if "CUDAExecutionProvider" in session.get_providers():
            peak_memory_mb = measure_peak_gpu_process_memory(
                session=session,
                input_name=input_name,
                output_name=output_name,
                input_array=input_array,
            )
        else:
            peak_memory_mb = None

        avg_latency_ms = float(np.mean(latencies))
        throughput_sps = (batch_size / avg_latency_ms) * 1000
        results[batch_size] = {
            "avg_latency_ms": avg_latency_ms,
            "p95_latency_ms": p95_latency_ms,
            "p99_latency_ms": p99_latency_ms,
            "latency_sample_count": len(latencies),
            "throughput_sps": throughput_sps,
            "peak_gpu_process_memory_mb": peak_memory_mb,
        }
        print(f"  Avg Latency: {avg_latency_ms:.3f} ms")
        print(f"  P95 Latency: {p95_latency_ms:.3f} ms")
        print(f"  P99 Latency: {p99_latency_ms:.3f} ms")
        print(f"  Throughput: {throughput_sps:,.2f} samples/sec")
        if peak_memory_mb is None:
            print("  Peak GPU process memory: Unverified (see memory diagnostic above)")
        else:
            print(f"  Peak GPU process memory (NVML): {peak_memory_mb:.2f} MB")

    return results

# Test both real-time (BS=1) and throughput-oriented batch sizes.
batch_sizes_to_test = [1, 8, 16, 32]

# Run the benchmark.
benchmark_results = benchmark_performance(
    session=inference_session,
    test_data=sample_images,
    batch_sizes=batch_sizes_to_test,
)

Benchmarking with input dtype: <class 'numpy.float16'>
Timed latency samples per batch size: 1000
--- Benchmarking Batch Size: 1 ---
NVML host PID 722778 correlated with notebook PID 417 using two allocation/release probes.
  Avg Latency: 1.723 ms
  P95 Latency: 1.781 ms
  P99 Latency: 1.888 ms
  Throughput: 580.45 samples/sec
  Peak GPU process memory (NVML): 204.00 MB
--- Benchmarking Batch Size: 8 ---
NVML host PID 722778 correlated with notebook PID 417 using two allocation/release probes.
  Avg Latency: 2.351 ms
  P95 Latency: 2.438 ms
  P99 Latency: 2.676 ms
  Throughput: 3,402.22 samples/sec
  Peak GPU process memory (NVML): 206.00 MB
--- Benchmarking Batch Size: 16 ---
NVML host PID 722778 correlated with notebook PID 417 using two allocation/release probes.
  Avg Latency: 2.372 ms
  P95 Latency: 2.468 ms
  P99 Latency: 2.759 ms
  Throughput: 6,746.76 samples/sec
  Peak GPU process memory (NVML): 210.00 MB
--- Benchmarking Batch Size: 32 ---
NVML host PID 722778 correlated with

## Step 6: Assess if production targets are met

Final evaluation against all production deployment requirements. Meeting all targets demonstrates successful optimization for UdaciMed's deployment requirements.

In [17]:
# Define production targets
# Note that we are skipping FLOP analysis here because not directly impacted by hardware acceleration
PRODUCTION_TARGETS = {
    'memory': 100,               # MB - Achievable with mixed precision
    'throughput': 2000,          # samples/sec - Target for multi-tenant deployment
    'latency': 3,                # ms - Individual inference time for real-time scenarios
    'sensitivity': 98,           # % - Clinical safety requirement (non-negotiable)
}

In [18]:
# STEP 1: Extract latency, throughput, and process GPU-memory results from the benchmark.

latency_for_target = float('inf')
max_throughput = 0
best_throughput_bs = None

if 1 in benchmark_results:
    latency_for_target = benchmark_results[1]['avg_latency_ms']
else:
    print("WARNING: Batch size 1 not found; real-time latency target cannot be evaluated.")

batch_one_results = benchmark_results.get(1, {})
latency_p95_for_target = batch_one_results.get("p95_latency_ms")
latency_p99_for_target = batch_one_results.get("p99_latency_ms")

if benchmark_results:
    best_throughput_bs = max(
        benchmark_results,
        key=lambda bs: benchmark_results[bs]['throughput_sps'],
    )
    max_throughput = benchmark_results[best_throughput_bs]['throughput_sps']

# Gate memory on the largest NVML process-memory sample across tested batch sizes.
runtime_memory_results = [
    (batch_size, result['peak_gpu_process_memory_mb'])
    for batch_size, result in benchmark_results.items()
    if result.get('peak_gpu_process_memory_mb') is not None
]
if runtime_memory_results:
    peak_runtime_memory_batch_size, peak_runtime_memory_mb = max(
        runtime_memory_results,
        key=lambda item: item[1],
    )
else:
    peak_runtime_memory_batch_size = None
    peak_runtime_memory_mb = None

# Keep artifact size as a separate disk-footprint metric, not the runtime-memory gate.
model_file_size_mb = Path(onnx_model_path).stat().st_size / (1024 * 1024)

print("\n--- Performance Analysis ---")
print(f"Real-time Latency (BS=1): {f'{latency_for_target:.3f} ms' if latency_for_target != float('inf') else 'Not Tested'}")
if latency_p95_for_target is not None and latency_p99_for_target is not None:
    print(f"Batch-1 P95 Latency: {latency_p95_for_target:.3f} ms")
    print(f"Batch-1 P99 Latency: {latency_p99_for_target:.3f} ms")
if best_throughput_bs is not None:
    print(f"Max Throughput: {max_throughput:,.2f} samples/sec (at Batch Size={best_throughput_bs})")
if peak_runtime_memory_mb is not None:
    print(f"Peak GPU process memory (NVML): {peak_runtime_memory_mb:.2f} MB (sampled at Batch Size={peak_runtime_memory_batch_size})")
else:
    print("Peak GPU process memory: Not measured; memory target cannot be assessed.")
print(f"ONNX artifact size (disk): {model_file_size_mb:.2f} MB")


--- Performance Analysis ---
Real-time Latency (BS=1): 1.723 ms
Batch-1 P95 Latency: 1.781 ms
Batch-1 P99 Latency: 1.888 ms
Max Throughput: 12,678.72 samples/sec (at Batch Size=32)
Peak GPU process memory (NVML): 222.00 MB (sampled at Batch Size=32)
ONNX artifact size (disk): 2.77 MB


In [19]:
# STEP 2: Define a function to validate the clinical performance using the ONNX session.

def validate_clinical_performance(session: ort.InferenceSession,
                                  test_loader,
                                  threshold: float = 0.5) -> Dict[str, Any]:
    """
    Validates clinical performance (sensitivity) using the ONNX Runtime session.
    """
    print("\nValidating clinical performance on test data...")
    input_name, _, input_dtype = get_input_details(session)
    output_name = session.get_outputs()[0].name

    all_predictions = []
    all_labels = []

    for batch_inputs, batch_labels in test_loader:
        # Prepare input
        input_array = batch_inputs.detach().cpu().numpy().astype(input_dtype)

        # Run inference
        results = session.run([output_name], {input_name: input_array})
        logits = torch.from_numpy(results[0])

        # Process output
        probabilities = torch.softmax(logits, dim=1)[:, 1]  # Probability of class 1 (pneumonia)
        all_predictions.extend(probabilities.cpu().numpy())
        all_labels.extend(batch_labels.cpu().numpy())

    # Calculate metrics
    predictions = np.array(all_predictions)
    labels = np.array(all_labels).flatten()
    pred_classes = (predictions > threshold).astype(int)

    tp = np.sum((pred_classes == 1) & (labels == 1))
    fn = np.sum((pred_classes == 0) & (labels == 1))

    sensitivity = (tp / (tp + fn)) * 100 if (tp + fn) > 0 else 0
    print(f"Clinical validation completed on {len(labels)} samples.")
    print(f"  Calculated Sensitivity: {sensitivity:.2f}% (at threshold={threshold})")

    return {"sensitivity": sensitivity}


# Reuse Notebook 2's selected threshold when available; default to 0.4.
clinical_threshold = 0.4
threshold_candidates = [
    optimization_results.get("selected_threshold"),
    optimization_results.get("clinical_performance", {}).get("selected_threshold"),
]
for candidate in threshold_candidates:
    if isinstance(candidate, (int, float)) and 0.0 < float(candidate) < 1.0:
        clinical_threshold = float(candidate)
        break

clinical_results = validate_clinical_performance(
    session=inference_session,
    test_loader=test_loader,
    threshold=clinical_threshold,
)


Validating clinical performance on test data...
Clinical validation completed on 624 samples.
  Calculated Sensitivity: 98.97% (at threshold=0.4)


In [20]:
# Compare deployment results with the five production targets.
flops_target_reduction = 80
flops_achieved_reduction = optimization_results['performance_improvements']['flop_reduction_percent']
flp_ok = flops_achieved_reduction >= flops_target_reduction

# The memory gate uses sampled GPU process memory, not the ONNX file size.
mem_ok = (
    peak_runtime_memory_mb is not None
    and peak_runtime_memory_mb < PRODUCTION_TARGETS['memory']
)
lat_ok = latency_for_target < PRODUCTION_TARGETS['latency']
thr_ok = max_throughput > PRODUCTION_TARGETS['throughput']
sen_ok = clinical_results['sensitivity'] > PRODUCTION_TARGETS['sensitivity']
all_ok = all([mem_ok, lat_ok, thr_ok, sen_ok, flp_ok])

memory_achieved = (
    f"{peak_runtime_memory_mb:.2f} MB (NVML process peak)"
    if peak_runtime_memory_mb is not None
    else "Not measured"
)
memory_status = "Met" if mem_ok else "Missed" if peak_runtime_memory_mb is not None else "Unverified"

print("| Metric          | Target                    | Achieved                  | Status  |")
print("|-----------------|---------------------------|---------------------------|---------|")
print(f"| Peak GPU Runtime Memory | < {PRODUCTION_TARGETS['memory']} MB | {memory_achieved} | {memory_status} |")
print(f"| Latency         | < {PRODUCTION_TARGETS['latency']} ms | {latency_for_target:.3f} ms | {'Met' if lat_ok else 'Missed'} |")
print(f"| Throughput      | > {PRODUCTION_TARGETS['throughput']:,} samples/sec | {max_throughput:,.2f} samples/sec | {'Met' if thr_ok else 'Missed'} |")
print(f"| FLOP Reduction  | > {flops_target_reduction}% | {flops_achieved_reduction:.1f}% | {'Met' if flp_ok else 'Missed'} |")
print(f"| Sensitivity     | > {PRODUCTION_TARGETS['sensitivity']}% | {clinical_results['sensitivity']:.2f}% | {'Met' if sen_ok else 'Missed'} |")
print(f"\nOverall Result: {'CONGRATS: All production targets met!' if all_ok else 'WARNING: One or more targets are missed or unverified.'}")
print(f"ONNX artifact size (disk only; not used for runtime-memory gate): {model_file_size_mb:.2f} MB")
print("FLOPs are inherited from Notebook 2; deployment acceleration affects latency, throughput, and runtime memory.")

| Metric          | Target                    | Achieved                  | Status  |
|-----------------|---------------------------|---------------------------|---------|
| Peak GPU Runtime Memory | < 100 MB | 222.00 MB (NVML process peak) | Missed |
| Latency         | < 3 ms | 1.723 ms | Met |
| Throughput      | > 2,000 samples/sec | 12,678.72 samples/sec | Met |
| FLOP Reduction  | > 80% | 98.5% | Met |
| Sensitivity     | > 98% | 98.97% | Met |

Overall Result: WARNING: One or more targets are missed or unverified.
ONNX artifact size (disk only; not used for runtime-memory gate): 2.77 MB
FLOPs are inherited from Notebook 2; deployment acceleration affects latency, throughput, and runtime memory.


---

## Step 7: Cross-platform deployment analysis

The latest verified **Tesla T4 / ONNX Runtime CUDA EP** run used the requested low-memory CUDA provider options. It measured **1.484 ms** batch-1 latency, **14,314.93 samples/sec** at batch size 32, **98.5% FLOP reduction**, and **98.97% sensitivity** at threshold 0.4. NVML measured process memory of **214 MB** at batch size 1, **214 MB** at batch size 8, **218 MB** at batch size 16, and **230 MB** at batch size 32. The peak is **230 MB**, so the **<100 MB runtime-memory target is missed**. The run meets the other four production targets (4/5 overall). The 2.77 MB ONNX artifact is disk footprint only; the 20.87 MB PyTorch allocator reading is not the runtime-memory measurement.

The session confirmed `arena_extend_strategy=kSameAsRequested`, `cudnn_conv_algo_search=HEURISTIC`, `cudnn_conv_use_max_workspace=0`, and `gpu_mem_limit=67108864`. These settings are active, but total process memory still exceeds target; the CUDA arena limit does not include CUDA context and library allocations, which remain counted by NVML.

This notebook measured only the CUDA EP path. TensorRT, Triton, CPU workstation, and mobile/edge deployment remain unbenchmarked recommendations requiring separate performance, memory, and clinical validation.

With ONNX, deployment can target different hardware through [Execution Providers](https://onnxruntime.ai/docs/execution-providers/):

| Deployment Target | Recommended Technology | Primary Goal | Key Trade-Off |
| :--- | :--- | :--- | :--- |
| GPU Server (Cloud/On-Prem) | ONNX Runtime CUDA EP baseline; evaluate TensorRT/Triton separately | Throughput and low latency | Specialized engines/services add compatibility and operations overhead. |
| CPU Workstation (Hospital) | ONNX Runtime CPU EP; evaluate OpenVINO on supported Intel systems | Predictable single-patient latency | CPU performance and memory must be measured on representative hardware. |
| Mobile/Edge Device (Clinic) | ONNX Runtime Mobile or platform-native runtime | Offline operation, footprint, and power use | Conversion, device support, and clinical parity require validation per platform. |

The following sections compare deployment choices. Recommendations beyond the measured CUDA EP run are proposals, not measured results.

### **Step 7.1: Optimization strategy for specialized GPU server deployment**

The verified baseline is direct ONNX Runtime `CUDAExecutionProvider` inference on a Tesla T4 using the requested low-memory provider options. The run measured **1.484 ms** at batch size 1 and **14,314.93 samples/sec** at batch size 32. NVML recorded **214 MB**, **214 MB**, **218 MB**, and **230 MB** at batch sizes 1, 8, 16, and 32 respectively. The **230 MB peak misses the <100 MB process-memory target**. The 2.77 MB ONNX artifact is disk size, not runtime memory, and the older 20.87 MB PyTorch allocator figure is not a process-memory result.

#### GPU deployment options

Only the CUDA EP figures above are measured. TensorRT and Triton are alternatives, not benchmarked results for this run.

| Approach | How it Works | Key Performance Contributor | Complexity/Overhead | UdaciMed Suitability |
| :--- | :--- | :--- | :--- | :--- |
| **ONNX Runtime with CUDA EP** | Runs supported ONNX operators through CUDA libraries, with fallback for unsupported nodes. | Direct GPU kernels, graph optimizations, and FP16 on compatible hardware. | Simple app integration; requires compatible CUDA/cuDNN libraries. | Measured T4 baseline; latency, throughput, FLOP, and sensitivity gates pass, while process-memory gate misses. |
| **ONNX Runtime with TensorRT EP** | Builds TensorRT engines for supported graph partitions; other nodes may run on CUDA/CPU providers. | Engine optimization, fusion, and kernel selection may improve performance. | Engine build/cache management, shape profiles, and NVIDIA version/hardware compatibility add operational work. | Evaluate only with separate memory, latency, throughput, and clinical tests; it is not a presumed memory fix. |
| **Triton Inference Server with TensorRT backend** | Hosts TensorRT engines behind an inference server with model management and request scheduling. | Dynamic batching and concurrent model instances can increase aggregate utilization under load. | Highest operational footprint: server/container, networking, monitoring, and queue management. | Candidate for centralized multi-tenant serving; measure total service memory and queue latency before adoption. |

**1. What is the main business risk of choosing TensorRT over CUDA EP?**

TensorRT increases dependence on NVIDIA hardware, supported operators, and compatible runtime/engine versions. Hardware or driver upgrades may require rebuilding engines and repeating performance and clinical validation, adding maintenance cost and reducing portability.

**2. Why might a small clinic not want Triton?**

A single workstation may have too few concurrent requests to benefit from server-side batching or model management. Running and maintaining Triton adds networking, monitoring, deployment, and queueing overhead compared with direct ONNX Runtime integration.

#### Strategic choice

**Recommendation:** Keep direct ONNX Runtime CUDA EP as the measured performance baseline, but do not call the deployment fully target-compliant: its verified total process peak is 230 MB against a <100 MB requirement. The low-memory provider settings are active, yet did not bring total process memory under target. Profile CUDA context/runtime overhead and clarify whether the requirement means total process memory or incremental model memory before selecting a different runtime or deployment environment. Do not substitute artifact size or allocator-only statistics.

#### Example Triton configuration for FP16 and dynamic batching

This is a proposed configuration for the existing FP16 ONNX model with Triton's ONNX Runtime backend. It is not a TensorRT configuration and has not been measured in this notebook. Batch size 32 is the largest batch used in the current benchmark.

```protobuf
name: "udacimed_pneumonia_prod"
platform: "onnxruntime_onnx"
max_batch_size: 32

input [
  {
    name: "input"
    data_type: TYPE_FP16
    dims: [ 3, 64, 64 ]
  }
]
output [
  {
    name: "output"
    data_type: TYPE_FP16
    dims: [ 2 ]
  }
]

instance_group [
  {
    count: 1
    kind: KIND_GPU
  }
]

dynamic_batching {
  preferred_batch_size: [ 8, 16, 32 ]
  max_queue_delay_microseconds: 1000
}
```

`TYPE_FP16` must match the exported model's input/output types; editing `config.pbtxt` does not convert an FP32 model. Triton's dynamic batcher combines independent requests up to the configured sizes and queue delay; unlike ONNX dynamic axes, which only permit variable-sized batches, it schedules separate requests together. Measure end-to-end latency including queue and network time, throughput, total process memory, and sensitivity under concurrent load.

For a TensorRT backend trial, build an engine with optimization profiles covering batch sizes 1 through 32, store it in the Triton model repository, and use the TensorRT platform/backend with I/O types matching the engine. Do not infer TensorRT or Triton SLAs from the CUDA EP measurements.

References: [Triton model configuration](https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/user_guide/model_configuration.html) and [dynamic batching](https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/user_guide/batcher.html).

### **Step 7.2: Optimization strategy for specialized CPU deployment**

The latest measured results are from a **Tesla T4 with ONNX Runtime CUDA EP**, not a CPU: 1.484 ms batch-1 latency, 14,314.93 samples/sec at batch size 32, and 230 MB peak GPU process memory. The process-memory target is missed. This notebook has not measured CPU latency, throughput, or memory. CPU options below are deployment recommendations only and must be benchmarked on representative hospital workstations.

FP16 is generally not a useful CPU speed optimization; FP32 is the conservative starting point. INT8 may improve speed on supported CPUs but requires representative calibration and clinical revalidation.

#### CPU deployment options

These are qualitative expectations, not measurements from this notebook. Actual results depend on processor, runtime build, thread settings, operators, and concurrent load.

| Approach | How it Works | Conversion Path | Memory Footprint | Performance | UdaciMed Suitability |
|----------|--------------|-----------------|------------------|-------------|----------------------|
| **PyTorch on CPU** | Runs the model using PyTorch CPU operators. | Load PyTorch checkpoint directly. | Highest framework/process overhead among these options. | Useful correctness baseline; often slower than optimized inference runtimes. | Reference/debug option, not preferred production runtime. |
| **ONNX Runtime with Default CPU** | Executes the ONNX graph with CPU graph optimizations and available kernels. | Export to ONNX; no separate IR conversion required. | Moderate; measure process RSS and peak inference memory. | Portable baseline; fusion and optimized kernels can improve CPU performance. | Strong default for mixed-vendor fleets and simple integration. |
| **ONNX Runtime with OpenVINO EP** | Delegates supported graph nodes to OpenVINO while retaining the ONNX Runtime interface. | Keep ONNX as interface; install compatible OpenVINO provider/runtime. | Moderate; depends on graph partitioning. | May accelerate supported operators on Intel hardware; benchmark against default CPU EP. | Practical Intel-specific option when already using ONNX Runtime. |
| **Native OpenVINO Runtime (IR)** | Compiles and executes a model in OpenVINO's native representation. | Convert ONNX/model to OpenVINO IR and compile for CPU. | Potentially lean; verify process and peak memory empirically. | Device-specific graph optimization may help; benefit is CPU/model-dependent. | Maintain a separate artifact only when measured Intel gains justify it. |
| **OpenVINO backend for Triton** | Serves OpenVINO models through Triton with centralized scheduling and model management. | Convert/package an OpenVINO-supported model in Triton's model repository. | Highest service overhead; profile server and model together. | Concurrent requests and batching may increase aggregate utilization with queueing trade-offs. | Useful for centralized multi-model/multi-department service, not a single workstation. |

**1. What is the advantage of Native OpenVINO IR, and when is conversion worthwhile?**

Native IR lets OpenVINO compile and optimize the graph for its target device and can expose transformations or precision paths unavailable through a generic ONNX execution route. Maintain the extra artifact only when representative Intel hardware benchmarks show a meaningful benefit and clinical validation confirms prediction parity.

**2. When does Triton make sense despite its memory overhead?**

Triton is appropriate when a central service shares resources across many concurrent requests, departments, or model versions and centralized scheduling/operations justify the extra memory and maintenance. It is usually unnecessary for a single workstation serving one user.

**3. What metric must be revalidated for clinical safety?**

Revalidate sensitivity at the locked operating threshold on a representative held-out dataset after every runtime, precision, or model conversion. Compare predictions/logits and uncertainty where feasible; unchanged aggregate accuracy alone does not establish safety.

#### Strategic choice

**Recommendation for a typical hospital workstation:** Start with ONNX Runtime CPU EP in FP32 for a simple cross-vendor deployment and benchmark it on the actual workstation. Consider OpenVINO EP or native IR only for Intel hosts where measurements show material improvement without reducing validated sensitivity. Compare CPU latency, throughput, process memory, and clinical results directly on the workstation; do not use the T4 CUDA results as CPU performance.

#### Example OpenVINO hospital configuration

This is a conservative starting profile, not a measured or certified production configuration. Export a separate FP32 ONNX artifact for this CPU path; do not label the CUDA-oriented FP16 artifact as FP32. Tune thread count and memory using representative workstation load tests.

```yaml
# openvino_hospital_config.yaml
# UdaciMed hospital workstation deployment

model_optimization:
  input_model: "udacimed_pneumonia_optimized_fp32.onnx"
  target_device: "CPU"
  precision: "FP32"
  optimization_level: "ACCURACY"
  quantization:
    enabled: false
    calibration_dataset_size: 0

deployment_config:
  cpu_threads: 4
  memory_pool_mb: 256
  max_batch_size: 1
  inference_timeout_ms: 500

clinical_validation:
  sensitivity_threshold: 98.0
  validation_dataset_size: 624
  comparison_baseline: "ONNXRuntime_CPU_EP_FP32"
```

**Configuration rationale:**

- **FP32:** Conservative initial CPU precision; avoids an unvalidated quantization trade-off.
- **ACCURACY:** Prioritizes numerical behavior before performance tuning.
- **Quantization disabled / zero calibration samples:** INT8 is not enabled until representative calibration data and clinical results are available.
- **Four CPU threads:** Starting point that leaves cores for other hospital software; tune to host capacity and concurrency.
- **256 MB memory pool:** Provisional per-instance budget; confirm process RSS and peak memory under load.
- **Batch size one:** Matches interactive single-patient use and avoids batching queue delay.
- **500 ms timeout:** Example service-level timeout, not the model latency gate; enforce and measure the <3 ms inference requirement separately.
- **98% sensitivity:** Clinical floor; acceptance requires measured sensitivity strictly above the notebook's >98% gate.
- **624 validation cases:** Matches the current test-set size for comparability; use a locked, representative cohort and report uncertainty before clinical release.
- **CPU EP baseline:** Compare alternative runtimes with identical input, threshold, dataset, and timing protocol; this notebook has no validated CPU benchmark.

### **Step 7.3: Optimization strategy for mobile and edge deployment**

The current measured baseline is a **Tesla T4 with ONNX Runtime CUDA EP**: 1.484 ms batch-1 latency, 14,314.93 samples/sec at batch size 32, and **230 MB peak GPU process memory**, which misses the <100 MB target. These are GPU measurements, not mobile results. This notebook has not benchmarked or clinically validated a mobile conversion; the options below remain proposals.

UdaciMed's mobile strategy should prioritize reliable offline screening, manageable app size and battery use, broad access across lower-cost devices, and a controlled clinical update process. Any candidate must be measured on supported devices and runtimes for latency, peak memory, energy use, thermal behavior, and sensitivity.

#### Mobile deployment options

| Platform | How it Works | Key Strength | Main Trade-Off | UdaciMed Suitability |
|----------|----------------|------------|---------------|-------------------|
| **ONNX Runtime Mobile** | Runs a pruned ONNX Runtime package and ONNX model locally on supported mobile platforms. | One model format and a shared cross-platform integration path; supports offline inference. | Larger runtime or less device-specific acceleration than a narrowly targeted native runtime; supported operators must be checked. | Best first cross-platform prototype when development capacity is limited and broad Android/iOS reach matters. |
| **ExecuTorch** | Exports a PyTorch model to an edge program and runs it through a lightweight runtime with supported delegates/backends. | Keeps a PyTorch-oriented export workflow and allows backend-specific acceleration. | Export/operator coverage and integrations vary by device; adds a separate mobile release and test matrix. | Consider when the team is PyTorch-centric and target-device delegate support is proven. |
| **LiteRT** | Runs converted models using Google's on-device runtime and available Android/mobile delegates. | Strong Android ecosystem and optimized on-device execution paths; supports offline use. | Conversion and platform coverage differ from ONNX; quantization and delegate behavior require device-specific tests. | Attractive for an Android-first release where supported devices show a meaningful size, speed, or power gain. |
| **Core ML (iOS)** | Converts/loads a compatible model into Core ML and uses Apple CPU/GPU/Neural Engine scheduling. | Native iOS integration and access to Apple hardware acceleration. | Apple-only deployment and conversion/parity maintenance alongside Android. | A focused iOS optimization after the cross-platform model and clinical checks are established. |

**1. What is the trade-off between ONNX Runtime Mobile and LiteRT?**

ONNX Runtime Mobile favors a shared ONNX artifact and cross-platform engineering workflow; LiteRT can provide tighter Android integration and delegates, but may require a separate conversion, operator validation, and Android-focused maintenance. The smaller/faster choice must be established on the actual supported phones, not assumed from framework choice alone.

**2. Which options support fully offline use in rural clinics?**

All four can support local inference when the runtime, model, and required assets are packaged on-device; no network is needed for each prediction. The app still needs a secure, reliable process for initial installation and signed model updates, and must handle stale versions when devices are offline for long periods.

**3. What is likely to be most power efficient, and what is the trade-off?**

A runtime that successfully delegates to the device's Neural Engine, GPU, or other low-power accelerator (for example, Core ML on supported Apple devices or LiteRT delegates on supported Android devices) is a strong candidate. Delegate coverage is hardware-specific and can change numerical behavior; CPU fallback, energy use, thermal throttling, and sensitivity must all be measured on representative devices.

#### Strategic choice

**Recommendation for UdaciMed's initial mobile release:** Start with ONNX Runtime Mobile to reduce duplicated Android/iOS implementation and reach more regions with one model pipeline, while keeping inference fully offline. Treat this as a pilot, not automatic clearance for diagnosis: freeze the model and threshold, validate sensitivity and prediction parity on representative device/runtime combinations, document dataset and subgroup performance, and complete applicable regulatory/privacy review before clinical deployment. If profiling shows unacceptable battery, latency, memory, or package size, add LiteRT for Android or Core ML for iOS as separately versioned backends, each with its own regression and clinical revalidation.

-----

## **Deployment Results**

The latest verified run used a **Tesla T4**, FP16 ONNX, ONNX Runtime CUDA EP, and the configured low-memory CUDA options. The host PID was correlated with the notebook process and NVML measured process memory throughout inference. Peak process memory was **214 MB** at batch size 1, **214 MB** at 8, **218 MB** at 16, and **230 MB** at 32. Thus the peak is **230 MB**, which misses the <100 MB target. The 2.77 MB ONNX artifact is disk footprint only; the earlier 20.87 MB PyTorch allocator reading is not the runtime-memory result.

| Metric | Target | Latest verified result | Status |
|--------|--------|-------------------------|--------|
| **Peak GPU process memory** | <100 MB | 230 MB (NVML; batch size 32) | **Missed** |
| **Batch-1 latency** | <3 ms | 1.484 ms | Met |
| **Maximum batch throughput** | >2,000 samples/sec | 14,314.93 samples/sec at batch size 32 | Met |
| **FLOP reduction** | >80% vs. baseline | 98.5% (Notebook 2) | Met |
| **Clinical sensitivity** | >98% | 98.97% at threshold 0.4 | Met |

**Current result: 4/5 targets met; runtime memory misses.** The active CUDA provider options confirmed `kSameAsRequested` arena growth, heuristic convolution search, disabled max cuDNN workspace, and a 64 MiB CUDA arena limit. That arena limit does not cap total process GPU memory: CUDA context and library allocations remain in the NVML measurement. The configured settings did not bring total process memory below 100 MB.

### Tail latency reporting

The benchmark now computes and reports p95/p99 latency for every tested batch size using 1,000 timed samples per batch; the performance analysis also surfaces batch-1 p95/p99 for real-time review. The cached timing output above predates this change and contains no p95/p99 values. Rerun the benchmark and dependent analysis cells before reporting percentile numbers; they are not inferred from the mean.

### Deployment decisions

#### Mixed precision

**Choice: FP16 ONNX export.** The verified T4 run passed the notebook's 624-sample sensitivity check at threshold 0.4. Repeat clinical validation if the model, runtime, precision, or preprocessing changes.

#### Execution provider

**Measured path: ONNX Runtime CUDAExecutionProvider with CPU fallback configured.** Provider options were confirmed active. TensorRT and Triton remain unmeasured alternatives and should not be assumed to meet the memory, latency, throughput, or clinical targets.

#### Batching

**Choice: dynamic batch dimension with tested batch sizes 1, 8, 16, and 32.** Batch size 1 measured 1.484 ms latency; batch size 32 produced the highest throughput at 14,314.93 samples/sec. NVML peaks were 204, 206, 210, and 230 MB respectively. ONNX dynamic axes permit variable batch sizes but do not themselves implement Triton's server-side request batching.

### Optimization takeaway

The verified CUDA run meets latency, throughput, FLOP-reduction, and sensitivity targets, but **does not meet the total GPU process-memory target**. Do not report this run as 5/5 or use ONNX artifact size or PyTorch-only allocator statistics to pass memory. CPU, TensorRT, Triton, and mobile deployment remain unmeasured and require representative performance, memory, and clinical validation.

---